# EDA & Feature Engineering

**Input:** `postings.csv` (LinkedIn Job Postings, Kaggle `arshkon/linkedin-job-postings`)

**Output:** `linkedin_job_postings_after_eda_and_feature_engineering.csv`

In [1]:
# Environment log — saved to file for reproducibility
import sys, platform, datetime, subprocess, shutil, json

env_log = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {},
}

if shutil.which("nvidia-smi"):
    env_log["gpu"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()

if shutil.which("nvcc"):
    env_log["cuda"] = subprocess.check_output(
        ["nvcc", "--version"], text=True).strip().split("\n")[-1]

try:
    import psutil
    env_log["ram_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
except ImportError:
    pass

for pkg in ["pip", "pandas", "numpy", "re", "sentence_transformers", "hashlib"]:
    try:
        m = __import__(pkg)
        env_log["packages"][pkg] = getattr(m, "__version__", "installed")
    except ImportError:
        env_log["packages"][pkg] = "NOT INSTALLED"

# Save to file
ENV_FILE = "environment_log.json"
with open(ENV_FILE, "w") as f:
    json.dump(env_log, f, indent=2)

print(json.dumps(env_log, indent=2))
print(f"\nSaved to {ENV_FILE}")

{
  "timestamp": "2026-03-17T13:38:22.170331+00:00",
  "python": "3.12.12",
  "platform": "Linux-6.6.113+-x86_64-with-glibc2.35",
  "packages": {
    "pip": "24.1.2",
    "pandas": "2.2.2",
    "numpy": "2.0.2",
    "re": "2.2.1",
    "sentence_transformers": "5.2.3",
    "hashlib": "installed"
  },
  "ram_gb": 12.7
}

Saved to environment_log.json


In [2]:
# ============================================================
# INPUT FILE INTEGRITY CHECK (SHA-512)
# ============================================================
import hashlib

EXPECTED_SHA512 = "52c066f2a0ff38e5a932fa54fdb14ebb1b576c5064c6a20db114eb00025bc1a49f9e1b82ff64d1ecb6e539bd41e472cfd5ae90d17f68298be741f05ca365bddc"

sha512 = hashlib.sha512()
with open("postings.csv", "rb") as f:
    for block in iter(lambda: f.read(8192), b""):
        sha512.update(block)

actual = sha512.hexdigest()

if actual == EXPECTED_SHA512:
    print(f"   SHA-512 MATCH — input file is intact")
    print(f"   {actual}")
else:
    print(f"   SHA-512 MISMATCH — input file may be corrupted or wrong!")
    print(f"   Expected: {EXPECTED_SHA512}")
    print(f"   Got:      {actual}")
    raise ValueError("Input file SHA-512 does not match. Aborting.")

   SHA-512 MATCH — input file is intact
   52c066f2a0ff38e5a932fa54fdb14ebb1b576c5064c6a20db114eb00025bc1a49f9e1b82ff64d1ecb6e539bd41e472cfd5ae90d17f68298be741f05ca365bddc


In [3]:
import pandas as pd
import numpy as np
import re

In [4]:
df = pd.read_csv("postings.csv")
print("Initial shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(2)

Initial shape: (123849, 31)
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0


In [5]:
# Remove duplicates
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)
df = df.drop_duplicates()
print("Shape after duplicate removal:", df.shape)

Duplicate rows: 0
Shape after duplicate removal: (123849, 31)


In [6]:
# Drop rows missing required columns
required_cols = ["title", "description", "formatted_experience_level"]
print("Missing values before cleaning:")
print(df[required_cols].isnull().sum())

df = df.dropna(subset=required_cols)

for col in required_cols:
    df = df[df[col].astype(str).str.strip() != ""]

print("Shape after dropping missing/empty:", df.shape)

Missing values before cleaning:
title                             0
description                       7
formatted_experience_level    29409
dtype: int64
Shape after dropping missing/empty: (94440, 31)


In [7]:
# Drop non-USD salary rows
# Salary thresholds (50K, 100K) assume USD — non-USD currencies would be misclassified
if "currency" in df.columns:
    non_usd = df["currency"].notna() & (df["currency"].str.strip().str.upper() != "USD")
    print(f"Dropping {non_usd.sum()} non-USD rows")
    df = df[~non_usd].reset_index(drop=True)
    print(f"Shape after dropping non-USD: {df.shape}")

Dropping 12 non-USD rows
Shape after dropping non-USD: (94428, 31)


In [8]:
# Filter by description length (400-700 words)
df["desc_word_count"] = df["description"].astype(str).str.split().str.len()
print("Description word count stats:")
print(df["desc_word_count"].describe())

df = df[(df["desc_word_count"] >= 400) & (df["desc_word_count"] <= 700)]
print("\nShape after description-length filtering:", df.shape)

Description word count stats:
count    94428.000000
mean       539.201709
std        300.561532
min          1.000000
25%        320.000000
50%        494.000000
75%        709.000000
max       3400.000000
Name: desc_word_count, dtype: float64

Shape after description-length filtering: (36253, 32)


In [9]:
# Map experience levels to 3 classes
def map_experience_level(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if any(term in x for term in ["intern", "entry", "junior", "associate", "trainee"]):
        return "junior"
    elif any(term in x for term in ["mid", "intermediate"]):
        return "mid"
    elif any(term in x for term in ["senior", "lead", "manager", "director", "executive", "principal", "head", "vp"]):
        return "senior"
    else:
        return np.nan

df["exp_level_3"] = df["formatted_experience_level"].apply(map_experience_level)

print("Original labels:")
print(df["formatted_experience_level"].value_counts(dropna=False).head(20))
print("\nMapped labels:")
print(df["exp_level_3"].value_counts(dropna=False))

df = df.dropna(subset=["exp_level_3"]).reset_index(drop=True)
print(f"\nShape after dropping unmapped levels: {df.shape}")

Original labels:
formatted_experience_level
Entry level         15873
Mid-Senior level    15104
Associate            2958
Director             1236
Internship            676
Executive             406
Name: count, dtype: int64

Mapped labels:
exp_level_3
junior    19507
mid       15104
senior     1642
Name: count, dtype: int64

Shape after dropping unmapped levels: (36253, 33)


In [10]:
# Salary signal
salary_col = "normalized_salary" if "normalized_salary" in df.columns else None

def salary_signal(x):
    if pd.isna(x):
        return "unknown"
    try:
        x = float(x)
    except:
        return "unknown"
    if x < 50000:
        return "junior"
    elif x <= 100000:
        return "mid"
    else:
        return "senior"

if salary_col:
    df["salary_signal"] = df[salary_col].apply(salary_signal)
else:
    df["salary_signal"] = "unknown"

print(df["salary_signal"].value_counts(dropna=False))

salary_signal
unknown    26993
mid         3536
senior      3055
junior      2669
Name: count, dtype: int64


In [11]:
# Clean skills text
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9,+/#.\-\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

if "skills_desc" in df.columns:
    df["skills_desc_clean"] = df["skills_desc"].apply(clean_text)
else:
    df["skills_desc_clean"] = np.nan

# Skill count
def count_skills(text):
    if pd.isna(text) or not str(text).strip():
        return 0
    parts = re.split(r",|;|\||\n", str(text))
    parts = [p.strip() for p in parts if p.strip()]
    parts = list(dict.fromkeys(parts))
    return len(parts)

df["skill_count"] = df["skills_desc_clean"].apply(count_skills)

def skill_count_band(n):
    if n == 0:
        return "unknown"
    elif n <= 3:
        return "low"
    elif n <= 7:
        return "medium"
    else:
        return "high"

df["skill_count_band"] = df["skill_count"].apply(skill_count_band)
print(df["skill_count_band"].value_counts(dropna=False))

skill_count_band
unknown    35557
low          614
medium        47
high          35
Name: count, dtype: int64


In [12]:
# Skill complexity signal
advanced_skills = [
    "machine learning", "deep learning", "nlp", "computer vision",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform",
    "spark", "hadoop", "airflow", "pytorch", "tensorflow",
    "mlops", "big data", "data engineering", "architecture",
    "microservices", "llm", "genai"
]

mid_skills = [
    "python", "sql", "java", "excel", "tableau", "power bi",
    "statistics", "data analysis", "git", "linux", "etl",
    "scikit-learn", "pandas", "numpy", "matplotlib"
]

def keyword_count(text, keywords):
    if pd.isna(text):
        return 0
    text = str(text).lower()
    return sum(1 for kw in keywords if kw in text)

df["advanced_skill_count"] = df["skills_desc_clean"].apply(lambda x: keyword_count(x, advanced_skills))
df["mid_skill_count"] = df["skills_desc_clean"].apply(lambda x: keyword_count(x, mid_skills))

def skill_signal(row):
    adv = row["advanced_skill_count"]
    midc = row["mid_skill_count"]
    total = row["skill_count"]
    if adv >= 2 or total >= 8:
        return "senior"
    elif adv == 1 or midc >= 2 or 4 <= total <= 7:
        return "mid"
    elif 1 <= total <= 3:
        return "junior"
    else:
        return "unknown"

signal_map = {"unknown": 0, "junior": 1, "mid": 2, "senior": 3}
df["skill_signal"] = df.apply(skill_signal, axis=1)
df["skill_signal_num"] = df["skill_signal"].map(signal_map)
print(df["skill_signal"].value_counts(dropna=False))

skill_signal
unknown    35557
junior       611
mid           50
senior        35
Name: count, dtype: int64


In [13]:
# Title signal
# FIX: returns "unknown" for unmatched titles (v1 defaulted to "mid")

def title_signal(title):
    title = str(title).lower()
    if any(w in title for w in ["intern", "trainee", "junior", "associate", "entry"]):
        return "junior"
    elif any(w in title for w in ["senior", "sr.", "sr ", "lead", "principal",
                                   "manager", "director", "head", "vp", "chief"]):
        return "senior"
    else:
        return "unknown"

df["title_signal"] = df["title"].apply(title_signal)
df["title_signal_num"] = df["title_signal"].map(signal_map)
print(df["title_signal"].value_counts(dropna=False))

title_signal
unknown    22571
senior     10466
junior      3216
Name: count, dtype: int64


In [14]:
# Years of experience from description
def extract_years(text):
    text = str(text).lower()
    matches = re.findall(r"(\d+)\+?\s*(?:years|year|yrs|yr)", text)
    if matches:
        return max(int(m) for m in matches)
    return 0

df["years_experience"] = df["description"].apply(extract_years)

def years_signal(y):
    if y == 0:
        return "unknown"
    elif y <= 2:
        return "junior"
    elif y <= 5:
        return "mid"
    else:
        return "senior"

df["years_signal"] = df["years_experience"].apply(years_signal)
df["years_signal_num"] = df["years_signal"].map(signal_map)
print(df["years_signal"].value_counts(dropna=False))

years_signal
unknown    15419
senior      9707
mid         7300
junior      3827
Name: count, dtype: int64


In [15]:
# Select and save final columns
final_cols = [
    "company_name", "title", "description", "location",
    "formatted_work_type", "formatted_experience_level",
    "skills_desc", "normalized_salary",
    "exp_level_3",
    "salary_signal",
    "skills_desc_clean", "skill_count", "skill_count_band",
    "advanced_skill_count", "mid_skill_count",
    "skill_signal", "skill_signal_num",
    "title_signal", "title_signal_num",
    "years_experience", "years_signal", "years_signal_num"
]
final_cols = [c for c in final_cols if c in df.columns]
df_final = df[final_cols].copy()

print("Final shape:", df_final.shape)
print("Columns:", df_final.columns.tolist())
print("\nLevel distribution:")
print(df_final["exp_level_3"].value_counts())
print("\nSignal distributions:")
for s in ["salary_signal", "skill_signal", "title_signal", "years_signal"]:
    if s in df_final.columns:
        print(f"\n{s}:")
        print(df_final[s].value_counts())

Final shape: (36253, 22)
Columns: ['company_name', 'title', 'description', 'location', 'formatted_work_type', 'formatted_experience_level', 'skills_desc', 'normalized_salary', 'exp_level_3', 'salary_signal', 'skills_desc_clean', 'skill_count', 'skill_count_band', 'advanced_skill_count', 'mid_skill_count', 'skill_signal', 'skill_signal_num', 'title_signal', 'title_signal_num', 'years_experience', 'years_signal', 'years_signal_num']

Level distribution:
exp_level_3
junior    19507
mid       15104
senior     1642
Name: count, dtype: int64

Signal distributions:

salary_signal:
salary_signal
unknown    26993
mid         3536
senior      3055
junior      2669
Name: count, dtype: int64

skill_signal:
skill_signal
unknown    35557
junior       611
mid           50
senior        35
Name: count, dtype: int64

title_signal:
title_signal
unknown    22571
senior     10466
junior      3216
Name: count, dtype: int64

years_signal:
years_signal
unknown    15419
senior      9707
mid         7300
junio

In [16]:
df_final.to_csv("linkedin_job_postings_after_eda_and_feature_engineering.csv", index=False)
print("Saved: linkedin_job_postings_after_eda_and_feature_engineering.csv")
print(f"Final rows: {len(df_final):,}")

Saved: linkedin_job_postings_after_eda_and_feature_engineering.csv
Final rows: 36,253


In [17]:
# ============================================================
# OUTPUT FILE SHA-512 CHECKSUM
# ============================================================
import hashlib

OUTPUT_FILE = "linkedin_job_postings_after_eda_and_feature_engineering.csv"

sha512_out = hashlib.sha512()
with open(OUTPUT_FILE, "rb") as f:
    for block in iter(lambda: f.read(8192), b""):
        sha512_out.update(block)

print(f"Output file: {OUTPUT_FILE}")
print(f"SHA-512:     {sha512_out.hexdigest()}")

Output file: linkedin_job_postings_after_eda_and_feature_engineering.csv
SHA-512:     c24cb643697a6067e0378fc3892191753f70b0f265cf8641ec6f6f4ee7503aa88e0ea0068e3a7c5c2006184e7dad641e6770115d678c98ab326ff116c73c6d3c
